# Linear regression on Samuel-3 data

Estimate the strengths of the effects of each variable on each other using linear regression.

We know that not all of our links are linear, but still.

Change certain columns to make the data simpler.


## Current setup

One-hot-encode the stroke team columns.

Remove rows with arrival-to-scan time at least 240 minutes or onset-to-arrival time at least 240 minutes. Note that this seems to remove a few of the stroke teams.

Define new features:

| Old column | New column | Description |
| --- | --- | --- |
| age | age_80plus| Age is at least 80 |
| stroke_severity | stroke_severity_mild | Stroke severity is less than 5 |
| stroke_severity | stroke_severity_moderate | Stroke severity is at least 5 and no more than 30 |
| stroke_severity | stroke_severity_severe | Stroke severity is more than 30 |
| discharge_disability | discharge_util | Discharge utility using Dijkland conversion from mRS score to utility |
| discharge_disability | discharge_mrsleqx for x from 0 to 5 inclusive | Discharge disability is less than x for x from 0 to 5 inclusive |

Final features: 'onset_during_sleep', 'precise_onset_known', 'atrial_fibrillation', 'afib_anticoagulant', 'prior_util', 'age_80plus', 'stroke_severity_mild', 'stroke_severity_moderate', 'stroke_severity_severe', 'arrival_to_scan_time',  2, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 18, 20, 21, 24, 25, 28, 29, 31, 32, 33, 34, 36, 38, 39, 40, 42, 43, 45, 47, 48, 49, 51, 55, 56, 58, 60, 61, 62, 65, 66, 68, 70, 71, 72, 73, 74, 76, 77, 78, 80, 82, 83, 85, 87, 93, 94, 95, 98, 99, 100, 103, 104, 106, 107, 108, 'thrombolysis', 'discharge_util'

Remove any row with missing values in any of the feature columns.

## Potential future setup

Haven't implemented this section yet


| Column | Original | Simple |
| --- | --- | --- |
| Age | 5-year age bands | Over/under 80 (i.e. age band 77.5 is under, 82.5 is over) |
| Prior disability | mRS 0-5 inclusive | mRS<=2 |
| Arrival-to-scan time | Continuous, rounded to nearest minute | Under 30 minutes |
| Stroke team | Categorical, 116 options | ? |
| Stroke severity | Continuous, 0-42 inclusive | Mild (0-9), moderate (10-25), severe (26-42) |
| Discharge disability | mRS 0-6 inclusive | mRS<=2 |

Conversion from mRS score to utility (References: Wang et al. 2020: Stroke. 2020;51:2411–2417; Dijkland et al. 2018: Stroke. 2018;49:965–971)
| Name | mRS=0 | mRS=1 | mRS=2 | mRS=3 | mRS=4 | mRS=5 | mRS=6 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Wang2020 | 0.97 | 0.88 | 0.74 | 0.55 | 0.2 | -0.19 | 0 |
| Dijkland2018 | 0.95 | 0.93 | 0.83 | 0.62 | 0.42 | 0.11 | 0 |



Always use these simplifications:
+ Convert mRS to utility

Do a few versions for various levels of simplification:
+ Stroke team excluded completely
+ Stroke severity simpler
+ Age over/under 80

## Code setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker  # for axis ticks
import os

import statsmodels.api as sm

## Load data

In [2]:
df = pd.read_csv('/home/anna/samuel3/data/cleaned_data_anon_teams(in).csv')

In [3]:
df['year'].describe()[['min', 'max']]

min    2020.0
max    2025.0
Name: year, dtype: float64

Relabel stroke teams:

In [4]:
make_teams = False
if make_teams:
    np.random.seed(42)
    
    team_keys = df['stroke_team'].unique()
    np.random.shuffle(team_keys)
    team_values = np.arange(len(team_keys))
    
    team_dict = dict(zip(team_keys, team_values))
    # Save a copy of this:
    pd.Series(team_dict).to_csv('./anon/teamlookup.csv')
else:
    team_dict = pd.read_csv('./anon/teamlookup.csv', index_col=0).squeeze().to_dict()

In [5]:
# Apply map to df:
df['stroke_team'] = df['stroke_team'].map(team_dict).astype(str)

One-hot-encode the stroke teams:

In [6]:
df = pd.concat((df, pd.get_dummies(df['stroke_team']).astype(int)), axis='columns')
df = df.drop('stroke_team', axis='columns')

Remove large arrival times:

In [7]:
df = df[df['onset_to_arrival_time'] < 60*4]

In [8]:
df['arrival_to_scan_time'].quantile(np.clip(np.arange(0.75, 1.01, 0.05), a_min=0.0, a_max=1.0))

0.75        46.0
0.80        57.0
0.85        75.0
0.90       115.0
0.95       216.0
1.00    132485.0
Name: arrival_to_scan_time, dtype: float64

In [9]:
df = df[df['arrival_to_scan_time'] < 4*60]

Make new column for discharge disability mRS<=x for x from 0 to 5 (note that mRS<=6 is always true):

In [10]:
for x in range(6):
    df[f'discharge_mrs{x}'] = (df['discharge_disability'] == x).astype(int)
    df[f'discharge_mrsleq{x}'] = (df['discharge_disability'] <= x).astype(int)
# for x in range(5):
#     df[f'prior_mrsleq{x}'] = (df['prior_disability'] <= x).astype(int)

In [11]:
cols_d = [c for c in df.columns if c.startswith('discharge')]
display(df.iloc[42:47][cols_d])

,discharge_destination,discharge_disability,discharge_mrs0,discharge_mrsleq0,discharge_mrs1,discharge_mrsleq1,discharge_mrs2,discharge_mrsleq2,discharge_mrs3,discharge_mrsleq3,discharge_mrs4,discharge_mrsleq4,discharge_mrs5,discharge_mrsleq5
132,community_team_or_esd,0.0,1,1,0,1,0,1,0,1,0,1,0,1
137,care_home,4.0,0,0,0,0,0,0,0,0,1,1,0,1
138,missing,NaN,0,0,0,0,0,0,0,0,0,0,0,0
140,community_team_or_esd,4.0,0,0,0,0,0,0,0,0,1,1,0,1
142,home,1.0,0,0,1,1,0,1,0,1,0,1,0,1


Convert mRS to utility (use the Dijkland values):

In [12]:
util_scores = dict(zip(range(7), [0.95, 0.93, 0.83, 0.62, 0.42, 0.11, 0.0]))

In [13]:
df['prior_util'] = df['prior_disability'].map(util_scores)
df['discharge_util'] = df['discharge_disability'].map(util_scores)

Convert stroke severity to mild/moderate/severe (cutoffs based on SHAP values from Pearn et al. 2023):

In [14]:
df['stroke_severity_mild'] = (df['stroke_severity'] < 5).astype(int)
df['stroke_severity_moderate'] = ((df['stroke_severity'] >= 5) & (df['stroke_severity'] <= 30)).astype(int)
df['stroke_severity_severe'] = (df['stroke_severity'] > 30).astype(int)

Convert age to over/under 80:

In [15]:
df['age_80plus'] = (df['age'] > 80).astype(int)

## Define linked variables

Set up which variables "cause" each other:

In [16]:
features = [
    'onset_during_sleep', 'precise_onset_known', 'atrial_fibrillation',
    'afib_anticoagulant',
    'prior_util',
    'age_80plus',
    # 'stroke_severity',
    'stroke_severity_mild', 'stroke_severity_moderate', 'stroke_severity_severe',
    'arrival_to_scan_time', 
    2, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 18, 20,
    21, 24, 25, 28, 29, 31, 32, 33, 34, 36, 38, 39, 40, 42, 43, 45, 47, 48,
    49, 51, 55, 56, 58, 60, 61, 62, 65, 66, 68, 70, 71, 72, 73, 74, 76, 77,
    78, 80, 82, 83, 85, 87, 93, 94, 95, 98, 99, 100, 103, 104, 106, 107, 108,
    # 'stroke_team',
    'thrombolysis', 'discharge_util',
    'discharge_mrsleq2'
]
# features += [f'prior_mrsleq{x}' for x in range(5)]
# features += [f'prior_mrs{x}' for x in range(5)]
# features += [f'discharge_mrsleq{x}' for x in range(6)]
# features += [f'discharge_mrs{x}' for x in range(6)]
features = [str(f) for f in features]

stroke_team_cols = [f for f in features if f.isnumeric()]

Check for missing values:

In [17]:
df[features].isna().sum(axis='rows').sort_values(ascending=False).head(5)

discharge_util         5779
precise_onset_known       0
onset_during_sleep        0
afib_anticoagulant        0
prior_util                0
dtype: int64

Remove any row with missing values:

In [18]:
mask = df[features].notna().all(axis='columns')
df = df[mask]

Check stroke team counts:

In [19]:
team_counts = {}
for team in stroke_team_cols:
    n_here = len(df[(df[team] > 0)])
    team_counts[team] = n_here

In [20]:
pd.Series(team_counts).sort_values()

62       83
51      220
6       255
18      279
107     298
       ... 
28     2161
60     2175
87     2249
66     2382
56     3358
Length: 66, dtype: int64

In [21]:
len(df[(df['age_80plus'] > 0)])

59550

In [22]:
len(df[~(df['age_80plus'] > 0)])

92794

## Calculate effects

Store links in here:

In [23]:
# feats = features + [f'odds_discharge_mrsleq{m}' for m in range(6)]

df_effects_bool = pd.DataFrame(columns=features, index=features, dtype=int)
# Fill missing data with zeros:
df_effects_bool = df_effects_bool.fillna(0)

In [24]:
edges = [
    # ('stroke_team', 'arrival_to_scan_time'), 
    # ('stroke_team', 'thrombolysis'),
    # ('stroke_team', 'discharge_disability'),
    # ('stroke_severity', 'thrombolysis'),
    # ('stroke_severity', 'discharge_util'),
    ('stroke_severity_mild', 'thrombolysis'),
    ('stroke_severity_mild', 'discharge_util'),
    ('stroke_severity_moderate', 'thrombolysis'),
    ('stroke_severity_moderate', 'discharge_util'),
    ('stroke_severity_severe', 'thrombolysis'),
    ('stroke_severity_severe', 'discharge_util'),
    ('arrival_to_scan_time', 'thrombolysis'),
    ('precise_onset_known', 'thrombolysis'),
    ('onset_during_sleep', 'precise_onset_known'),
    ('onset_during_sleep', 'thrombolysis'),
    ('atrial_fibrillation', 'afib_anticoagulant'),
    ('atrial_fibrillation', 'discharge_util'),
    ('afib_anticoagulant', 'thrombolysis'),
    ('age_80plus', 'prior_util'),
    ('age_80plus', 'thrombolysis'),
    ('age_80plus', 'discharge_util'),
    ('prior_util', 'thrombolysis'),
    ('prior_util', 'discharge_util'),
    ('thrombolysis', 'discharge_util'),
]
for t in stroke_team_cols:
    edges += [(t, 'arrival_to_scan_time'), 
              (t, 'thrombolysis'), (t, 'discharge_util')]
new_edges = []
for edge in edges:
    if edge[1] == 'discharge_util':
        # new_edges += [(edge[0], f'discharge_mrsleq{x}') for x in range(6)]
        new_edges += [(edge[0], 'discharge_mrsleq2')]
        # new_edges += [(edge[0], f'discharge_mrs{x}') for x in range(6)]
edges += new_edges

# mrs_edges = []
# for m in range(5):
#     mrs_edges += [
#         ('age_80plus', f'prior_mrsleq{m}'),
#         (f'prior_mrsleq{m}', 'thrombolysis'),
#         (f'prior_mrsleq{m}', 'discharge_util'),
#         ]
#     for n in range(6):
#         mrs_edges += [
#             (f'prior_mrsleq{m}', f'discharge_mrsleq{n}'),
#             ]
# edges += mrs_edges

# # Make outcomes talk to each other:
# out_edges = []
# for x in range(5, 0, -1):
#     for y in range(x-1, -1, -1):
#         out_edges.append((f'discharge_mrsleq{y}', f'discharge_mrsleq{x}'))
# edges += out_edges

In [25]:
# out_edges

In [26]:
for edge in edges:
    df_effects_bool.loc[edge[0], edge[1]] = 1

In [27]:
s = (df_effects_bool.sum(axis='rows') > 0)
nodes_with_causes = s.index[s]

In [28]:
df_effects_bool[nodes_with_causes]

,precise_onset_known,afib_anticoagulant,prior_util,arrival_to_scan_time,thrombolysis,discharge_util,discharge_mrsleq2
onset_during_sleep,1.0,0.0,0.0,0.0,1.0,0.0,0.0
precise_onset_known,0.0,0.0,0.0,0.0,1.0,0.0,0.0
atrial_fibrillation,0.0,1.0,0.0,0.0,0.0,1.0,1.0
afib_anticoagulant,0.0,0.0,0.0,0.0,1.0,0.0,0.0
prior_util,0.0,0.0,0.0,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...
107,0.0,0.0,0.0,1.0,1.0,1.0,1.0
108,0.0,0.0,0.0,1.0,1.0,1.0,1.0
thrombolysis,0.0,0.0,0.0,0.0,0.0,1.0,1.0
discharge_util,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Calculate link strengths:

In [29]:
# def outcome_calculator(out_col, df_data, df_file):
    
#     noise_out = np.random.normal(0, 0.1, len(df_data))

#     cols = [c for c in df_file.index if (c != 'const') & (c in df_data.columns)]

#     out = np.sum((
#         [noise_out] +
#         [df_data[col] * df_file.loc[col, out_col] for col in cols],
#     ), axis=0) + df_file.loc['const', out_col]

#     # out_bin = np.array([sigmoid(x) for x in out]).flatten()
#     return out#_bin

In [30]:
df_effects = pd.DataFrame(columns=features, index=features, dtype=float)
dict_results = {}
dict_parents = {}

for node in df_effects_bool.columns:
    parents = df_effects_bool[df_effects_bool[node] > 0].index
    if len(parents) < 1:
        # Nothing to calculate here.
        pass
    else:
        dict_parents[node] = parents
        # Set up Ordinary Least Squares linear regression:
        Y = df[node]
        X = df[parents]
        X = sm.add_constant(X)
        model = sm.OLS(Y,X)
        results = model.fit()
        dict_results[node] = results

        for parent in results.params.keys():
            df_effects.loc[parent, node] = results.params[parent]

        # if node.startswith('discharge') & ('mrsleq' in node):
        #     # Generate probabilities for this discharge mRS<=x.
        #     df[f'odds_{node}'] = outcome_calculator(node, df, df_effects)

In [31]:



# noise_out = np.random.normal(0, 0.1, len(df))

# cols = [c for c in df_effects.index if (c != 'const') & (c in df.columns)]

# out = np.sum((
#     [noise_out] +
#     [df[col] * df_effects.loc[col, node] for col in cols],
# ), axis=0) + df_effects.loc['const', node]


In [32]:
# for col in cols:
#     print(col, node, df_effects.loc[col, node])

In [33]:
# [df[col] * df_effects.loc[col, node] for col in cols]

In [34]:
# df_effects.loc['const', node]

In [35]:
# d = outcome_calculator(node, df, df_effects)

In [36]:
# d

In [37]:
# d[0]

In [38]:
# d[1]

In [39]:
# d[2]

In [40]:
# print(node, parents)

In [41]:
# df[parents]

In [42]:
df_effects[nodes_with_causes]

,precise_onset_known,afib_anticoagulant,prior_util,arrival_to_scan_time,thrombolysis,discharge_util,discharge_mrsleq2
onset_during_sleep,-0.658210,NaN,NaN,NaN,-0.121545,NaN,NaN
precise_onset_known,NaN,NaN,NaN,NaN,0.137368,NaN,NaN
atrial_fibrillation,NaN,0.634384,NaN,NaN,NaN,-0.026789,-0.024201
afib_anticoagulant,NaN,NaN,NaN,NaN,-0.298336,NaN,NaN
prior_util,NaN,NaN,NaN,NaN,0.323251,0.551239,0.779546
...,...,...,...,...,...,...,...
108,NaN,NaN,NaN,-6.274099,0.037439,-0.021214,-0.033155
thrombolysis,NaN,NaN,NaN,NaN,NaN,0.080275,0.104509
discharge_util,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discharge_mrsleq2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save results

In [43]:
df_effects[nodes_with_causes].round(5).to_csv(os.path.join('summary_data', 'effects_linreg_allpatients.csv'))

## View results

In [44]:
cols_discharge = [c for c in df_effects.columns if 'discharge' in c]
inds_discharge = [c for c in df_effects.index if 'discharge' in c]

df_effects.loc[inds_discharge, cols_discharge].fillna(0.0)

,discharge_util,discharge_mrsleq2
discharge_util,0.0,0.0
discharge_mrsleq2,0.0,0.0


In [45]:
df_effects[nodes_with_causes].describe()

,precise_onset_known,afib_anticoagulant,prior_util,arrival_to_scan_time,thrombolysis,discharge_util,discharge_mrsleq2
count,2.000000,2.000000,2.000000,67.000000,76.000000,74.000000,74.000000
mean,0.000152,0.337262,0.388963,-0.163547,-0.001248,0.003031,0.006030
std,0.931064,0.420194,0.717464,14.119253,0.082196,0.093573,0.124087
min,-0.658210,0.040139,-0.118361,-19.236074,-0.298336,-0.419159,-0.299238
25%,-0.329029,0.188700,0.135301,-7.969158,-0.034271,-0.027049,-0.044391
50%,0.000152,0.337262,0.388963,-0.289811,-0.010704,-0.005786,-0.013497
75%,0.329333,0.485823,0.642625,2.945668,0.021709,0.024047,0.067193
max,0.658514,0.634384,0.896287,84.800329,0.323251,0.551239,0.779546


In [46]:
for col in cols_discharge:
    print(col)
    display(df_effects[col].sort_values(ascending=False, key=lambda col: col.abs()).head())
    print('')

discharge_util


prior_util                  0.551239
stroke_severity_severe     -0.419159
const                       0.187256
stroke_severity_mild        0.124227
stroke_severity_moderate   -0.119434
Name: discharge_util, dtype: float64


discharge_mrsleq2


prior_util                0.779546
stroke_severity_severe   -0.299238
stroke_severity_mild      0.188509
6                        -0.184239
49                        0.170850
Name: discharge_mrsleq2, dtype: float64

In [47]:
df_effects.loc['arrival_to_scan_time', 'thrombolysis']

np.float64(-0.0017738153837382246)

In [48]:
df_effects.loc['thrombolysis', 'discharge_util']

np.float64(0.08027467487291265)

In [49]:
for node in nodes_with_causes:
    i = df_effects[node].abs().idxmax()
    print(f'{node:20s} {i:20s} {df_effects.loc[i, node]:10.5f}')

precise_onset_known  const                   0.65851
afib_anticoagulant   atrial_fibrillation     0.63438
prior_util           const                   0.89629
arrival_to_scan_time 62                     84.80033
thrombolysis         prior_util              0.32325
discharge_util       prior_util              0.55124
discharge_mrsleq2    prior_util              0.77955


In [50]:
dict_parents['arrival_to_scan_time']

Index(['2', '3', '5', '6', '7', '8', '9', '11', '12', '13', '14', '18', '20',
       '21', '24', '25', '28', '29', '31', '32', '33', '34', '36', '38', '39',
       '40', '42', '43', '45', '47', '48', '49', '51', '55', '56', '58', '60',
       '61', '62', '65', '66', '68', '70', '71', '72', '73', '74', '76', '77',
       '78', '80', '82', '83', '85', '87', '93', '94', '95', '98', '99', '100',
       '103', '104', '106', '107', '108'],
      dtype='object')

In [51]:
dict_results['thrombolysis'].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           thrombolysis   R-squared:                       0.225
Model:                            OLS   Adj. R-squared:                  0.224
Method:                 Least Squares   F-statistic:                     587.8
Date:                Wed, 12 Aug 2026   Prob (F-statistic):               0.00
Time:                        09:17:25   Log-Likelihood:                -81480.
No. Observations:              152344   AIC:                         1.631e+05
Df Residuals:                  152268   BIC:                         1.639e+05
Df Model:                          75                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                        0.0113      0.015      0.753      0.452      -0.018       0.041
onset_during_sleep          -0.1215      0.005    -22.238      0.000      -0.132      -0.111
precise_onset_known          0.1374      0.002     57.901      0.000       0.133       0.142
afib_anticoagulant          -0.2983      0.003   -103.375      0.000      -0.304      -0.293
prior_util                   0.3233      0.006     50.123      0.000       0.311       0.336
age_80plus                  -0.0156      0.002     -6.695      0.000      -0.020      -0.011
stroke_severity_mild        -0.0724      0.014     -5.283      0.000      -0.099      -0.046
stroke_severity_moderate     0.1837      0.014     13.451      0.000       0.157       0.210
stroke_severity_severe      -0.0433      0.017     -2.491      0.013      -0.077      -0.009
arrival_to_scan_time        -0.0018   2.78e-05    -63.705      0.000      -0.002      -0.002
2                           -0.0143      0.013     -1.093      0.274      -0.040       0.011
3                           -0.0826      0.013     -6.198      0.000      -0.109      -0.056
5                            0.0193      0.011      1.731      0.083      -0.003       0.041
6                           -0.0116      0.026     -0.449      0.654      -0.062       0.039
7                            0.1129      0.009     11.934      0.000       0.094       0.131
8                           -0.0265      0.009     -2.911      0.004      -0.044      -0.009
9                            0.0321      0.009      3.421      0.001       0.014       0.051
11                          -0.0153      0.011     -1.374      0.169      -0.037       0.007
12                          -0.0383      0.012     -3.176      0.001      -0.062      -0.015
13                           0.1355      0.011     12.825      0.000       0.115       0.156
14                           0.0031      0.011      0.274      0.784      -0.019       0.025
18                           0.0955      0.025      3.851      0.000       0.047       0.144
20                          -0.0125      0.014     -0.889      0.374      -0.040       0.015
21                          -0.0387      0.012     -3.349      0.001      -0.061      -0.016
24                          -0.0179      0.012     -1.509      0.131      -0.041       0.005
25                          -0.0099      0.011     -0.935      0.350      -0.031       0.011
28                           0.0720      0.009      7.961      0.000       0.054       0.090
29                          -0.1026      0.009    -10.889      0.000      -0.121      -0.084
31                           0.0860      0.014      6.220      0.000       0.059       0.113
32                          -0.0430      0.012     -3.726      0.000      -0.066      -0.020
33            

In [52]:
dir(dict_results['arrival_to_scan_time'])

['HC0_se',
 'HC1_se',
 'HC2_se',
 'HC3_se',
 '_HCCM',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abat_diagonal',
 '_cache',
 '_data_attr',
 '_data_in_cache',
 '_get_robustcov_results',
 '_get_wald_nonlinear',
 '_is_nested',
 '_transform_predict_exog',
 '_use_t',
 '_wexog_singular_values',
 'aic',
 'bic',
 'bse',
 'centered_tss',
 'compare_f_test',
 'compare_lm_test',
 'compare_lr_test',
 'condition_number',
 'conf_int',
 'conf_int_el',
 'cov_HC0',
 'cov_HC1',
 'cov_HC2',
 'cov_HC3',
 'cov_kwds',
 'cov_params',
 'cov_type',
 'df_model',
 'df_resid',
 'eigenvals',
 'el_test',
 'ess',
 'f_pvalue',
 'f_t

In [53]:
for k, v in dict_results.items():
    print(f'{k:>20s} {v.rsquared:.3f}')

 precise_onset_known 0.077
  afib_anticoagulant 0.462
          prior_util 0.106
arrival_to_scan_time 0.023
        thrombolysis 0.225
      discharge_util 0.359
   discharge_mrsleq2 0.287
